# 迭代

罗马不是一日建成的。同样，神经网络模型也不是一次训练就可以完成的。

前几章，我们每次都只是用一个样本训练一次。结果虽然方向正确，但损失值仍然很高，改进有限。

真正的模型训练遵循一个朴素的规则：利用**大量样本**，通过成千上万次**反复试错**，以**小步快跑**的方式逐渐逼近最佳参数。这种重复且不断改进的过程，称为**迭代**（Iteration）。

In [1]:
import numpy as np

## 数据集

模型训练所需的大量样本，统称为**数据集**（Dataset）。数据集中的每一条数据，称为一个**样本**（Sample）。每个样本都包含两部分：

* **特征值**（Feature）：输入数据，如天气情况；
* **标签值**（Label）：对应的真实结果，如实际销量。

在实践中，数据集通常会被随机划分为两部分：

* **训练集**（Training Set）：用于模型训练，通常占较大比例（如 80%）；
* **测试集**（Test Set）：用于评估训练效果，通常占较小比例（如 20%）。

划分的目的是保留一部分**模型从未见过的新数据**，用来检验模型是否真正学会了规律，还是只死记硬背了训练数据。这就像期末考试，老师不会用你做过的练习题出卷，而是用新题来检验你是否真正掌握了知识。

``💡 在深度学习中，数据集越大，模型通常能学到越通用的规律。真实项目中的数据集往往包含数万乃至数百万个样本。``

### 训练数据：特征、标签

小明提供了过去四天的天气记录和销售数据，构成训练集。

我们用 NumPy 二维数组保存，**每行代表一个样本**。

In [2]:
train_features = np.array([[22.5, 72.0],
                           [31.4, 45.0],
                           [19.8, 85.0],
                           [27.6, 63.0]])
train_labels = np.array([[95],
                         [210],
                         [70],
                         [155]])

### 测试数据：特征、标签

前几章一直使用的那条数据（温度 28.1°C、湿度 58%、销量 165），现在划入测试集。

这样，我们共有 5 个样本：4 个训练、1 个测试，比例恰好是 80% / 20%。

In [3]:
test_features = np.array([[28.1, 58.0]])
test_labels = np.array([[165]])

``💡 数据集中的数据以二维数组的形式保存，每行代表一个样本。以后，我们还会遇到以更高维度数组保存的数据，例如图像数据通常用三维数组（高度 × 宽度 × 颜色通道）表示。``

## 模型

模型结构与上一章完全相同，直接沿用。

### 参数：权重、偏置

In [4]:
weight = np.ones((1, 2)) / 2
bias = np.zeros(1)

### 推理函数

In [5]:
def forward(x, w, b):
    return x @ w.T + b

### 损失函数（均方误差）

In [6]:
def mse_loss(p, y):
    return np.mean(np.square(y - p))

### 梯度函数

In [7]:
def gradient(p, y):
    return -2 * (y - p)

### 反向函数

In [8]:
def backward(x, d, w, b, lr):
    w -= d * x * lr
    b -= d * lr
    return w, b

## 训练

### 超参数：学习率

In [9]:
LEARNING_RATE = 0.00001

### 迭代

每次将**一个样本**送入模型，完成以下三个步骤，称为**一次迭代**：

1. **前向传播**：用当前参数计算预测值；

```{figure} images/forward.png
:align: center
:width: 300px

* $x$：特征值；
* $w, b$：权重和偏置；
* $p$：预测值。

```

2. **梯度计算**：计算预测值与标签值之间的误差项；

```{figure} images/gradient.png
:align: center
:width: 260px

* $p$：预测值；
* $y$：标签值；
* $d$：误差项。

```

3. **反向传播**：根据误差项和学习率，更新参数。

```{figure} images/backward.png
:align: center
:width: 300px

* $d$：误差项；
* $x$：特征值；
* $w, b$：权重和偏置。

```

现在将训练集中的 4 个样本依次送入训练，完成 4 次迭代。

In [10]:
for i in range(len(train_features)):
    # 依次读入每个训练样本
    feature = train_features[i]
    label = train_labels[i]

    # 前向传播
    prediction = forward(feature, weight, bias)
    # 梯度计算
    delta = gradient(prediction, label)
    # 反向传播（更新参数）
    weight, bias = backward(feature, delta, weight, bias, LEARNING_RATE)

print(f'weight: {weight}')
print(f'bias:   {bias}')

weight: [[0.67678017 0.8307117 ]]
bias:   [0.0060984]


经过 4 次迭代（4 个样本），权重从初始的 `[0.5, 0.5]` 更新到了 `[0.677, 0.831]`，偏置也从 `0` 增至 `0.0061`。参数在向正确方向持续移动。

## 验证

模型训练完成后，用**测试集**（模型从未见过的新数据）评估效果。

``💡 注意：评估时只做前向传播，不进行梯度计算和参数更新。测试集的作用是检验，不是训练。``

### 推理

In [11]:
predictions = forward(test_features, weight, bias)
print(f'predictions: {predictions}')

predictions: [[67.20489976]]


### 评估

In [12]:
loss = mse_loss(predictions, test_labels)
print(f'loss: {loss:.4f}')

loss: 9563.8816


经过一轮迭代，测试集损失值从初始的 `14,871` 降至 `9,563`，下降了约 36%。

预测值也从初始的 `43` 提升到了 `67`，向真实值 165 又迈进了一步。但显然，4 次迭代还远不够，模型需要在更多样本上反复训练，才能逐步收敛。

## 课后练习

将训练数据改为逆序（样本 4 → 3 → 2 → 1）重新迭代一遍，损失值与顺序迭代相比有何变化？这说明迭代顺序对训练有影响吗？